In [ ]:
aws_region = "us-east-1"

In [ ]:
os.environ['RS_KB_ID'] = RS_KB_ID = 'xxxxxx'

In [ ]:
# model used for the structured data kb

model_id_novapro = "us.amazon.nova-pro-v1:0"
model_id = model_id_novapro
llm = model_id


In [ ]:
config = Config(
        retries = dict(
            max_attempts = 10,
            total_max_attempts = 25,
        )
    )

bedrock_client = boto3.client("bedrock-runtime", config=config) 

In [ ]:
#######
# Structured Data  RAG
#######
def rag_node_redshift(state: MultiAgentState):
    """
    RAG node function using Amazon Knowledge Bases for Redshift data retrieval
    Args:
        state: MultiAgentState containing the question
    Returns:
        dict: Contains the answer, response object, and image citations
    """
    try:
        # Initialize the Knowledge Base retriever
        retriever = AmazonKnowledgeBasesRetriever(
            knowledge_base_id=RS_KB_ID,
            retrieval_config={
                "vectorSearchConfiguration": {
                    "numberOfResults": 3
                }
            }
        )

        # Create the QA chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            retriever=retriever,
            return_source_documents=True
        )

        # Get the question from state
        question = state['question']

        # Get response from the chain
        response = qa_chain(question)
        
        # Extract the generated answer
        generation = response['result']
        
        # Update state with error information
        state['answer'] = f"Error occurred while processing the question: {str(e)}"
        state['raw_response'] = None
        return state  # Return the entire state object

In [ ]:
workflow.add_node("database_expert_agent", rag_node_redshift)